# Sparkle V2 — Inferencia end-to-end: de un video a la predicción

**Entorno:** Google Colab (CPU alcanza) · TensorFlow 2.20 + Keras 3 + MediaPipe
**Requiere:** la carpeta `modelo_final/` generada por `Sparkle_V2_Fase3_3_Modelo_Final.ipynb`.

Este notebook recorre el pipeline completo de producción: toma un **video de 10 segundos** y
devuelve la probabilidad de desatención, la clase predicha, el nivel ordinal y el mapa de calor
temporal. Es la referencia de implementación para integrar el modelo en la app Sparkle.

```
video (10 s)
   │
   ├─► MediaPipe FaceLandmarker  ──┬─► 22 features geométricas × 60 frames
   │    (una sola detección        │      → interpolación de NaN → +10 deltas → 32 features
   │     por frame)                │      → escalado
   │                               │
   │                               └─► crop facial 112×112 × 16 frames
   │                                      → MobileNetV2 (TFLite) → embeddings 128-d
   │                                      → escalado
   │
   └─► Modelo de decisión (TFLite) ──► P(desatención) · nivel ordinal · pesos de atención
```

**Optimización respecto de la extracción del dataset.** En Fase 2 y 3.2a el detector de MediaPipe
se corrió dos veces sobre cada video: una para las features geométricas y otra para los recortes.
Acá se ejecuta **una sola vez por frame** y el mismo resultado alimenta ambas ramas. Para
inferencia en dispositivo esa diferencia es sustancial.

**Contrato de privacidad.** Ni el frame ni el recorte se persisten en ningún momento: viven en
memoria mientras se calcula el embedding y se descartan. El único dato que sobrevive a la
inferencia es el vector de decisión.

In [ ]:
%pip install -q mediapipe

In [ ]:
# =====================================================================
# SETUP — artefactos del modelo y dependencias
# =====================================================================
import sys, os, glob, json, time, math

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
    WORK_DIR = '/content'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2'); WORK_DIR = '.'

MODEL_DIR = f'{PROJECT_DIR}/modelo_final'
assert os.path.isdir(MODEL_DIR), (
    f'Falta {MODEL_DIR}. Correr primero Sparkle_V2_Fase3_3_Modelo_Final.ipynb')

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
import urllib.request

print('Artefactos disponibles:')
for f in sorted(os.listdir(MODEL_DIR)):
    print(f'  {os.path.getsize(f"{MODEL_DIR}/{f}")/1024:9.1f} kB  {f}')

# --- Estadisticos de preprocesamiento: sin esto las predicciones son arbitrarias ---
prep = np.load(f'{MODEL_DIR}/preprocesamiento.npz', allow_pickle=True)
GEOM_MEDIANS = prep['geom_medians']
EMB_MEDIANS  = prep['emb_medians']
GEOM_MEAN, GEOM_SCALE = prep['geom_scaler_mean'], prep['geom_scaler_scale']
EMB_MEAN,  EMB_SCALE  = prep['emb_scaler_mean'],  prep['emb_scaler_scale']
FEATURE_NAMES   = list(prep['feature_names_22'])
DELTA_FEATURES  = list(prep['delta_features'])
N_FRAMES        = int(prep['n_frames'])
N_EMB_FRAMES    = int(prep['n_emb_frames'])
EMB_DIM         = int(prep['emb_dim'])
CROP_SIZE       = 112

print(f'\n{len(FEATURE_NAMES)} features geometricas | {N_FRAMES} frames | '
      f'{N_EMB_FRAMES} frames de apariencia | embeddings {EMB_DIM}-d')

# --- Modelo de landmarks ---
MODEL_PATH = f'{WORK_DIR}/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/face_landmarker/'
        'face_landmarker/float16/latest/face_landmarker.task', MODEL_PATH)
print(f'FaceLandmarker: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB')

## 1. Extracción de features — idéntica a la del entrenamiento

Cualquier divergencia entre cómo se calcularon las features al entrenar y cómo se calculan al
inferir degrada el modelo en silencio. Este bloque replica exactamente el código de Fase 2
(landmarks de EAR, MAR, ángulos de Euler y los 16 blendshapes curados) y el de Fase 3.2a
(recorte cuadrado con margen del 25 % y `preprocess_input` de MobileNetV2).

In [ ]:
# --- Landmarks e indices (identicos a Fase 2) ---
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
MOUTH_CORNERS  = [61, 291]
MOUTH_VERTICAL = [13, 14]
BLENDSHAPES_OF_INTEREST = [
    'eyeBlinkLeft', 'eyeBlinkRight', 'eyeSquintLeft', 'eyeSquintRight',
    'eyeLookInLeft', 'eyeLookOutLeft', 'eyeLookUpLeft', 'eyeLookDownLeft',
    'eyeLookInRight', 'eyeLookOutRight', 'eyeLookUpRight', 'eyeLookDownRight',
    'browDownLeft', 'browDownRight', 'browInnerUp', 'jawOpen',
]
assert FEATURE_NAMES == ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch', 'mar'] \
       + BLENDSHAPES_OF_INTEREST, 'El orden de features no coincide con el del entrenamiento'

def create_face_landmarker():
    base = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    opts = mp_vision.FaceLandmarkerOptions(
        base_options=base, running_mode=mp_vision.RunningMode.IMAGE, num_faces=1,
        min_face_detection_confidence=0.5,
        output_facial_transformation_matrixes=True, output_face_blendshapes=True)
    return mp_vision.FaceLandmarker.create_from_options(opts)

def eye_aspect_ratio(lm, ids, w, h):
    p = np.array([(lm[i].x * w, lm[i].y * h) for i in ids])
    horiz = np.linalg.norm(p[0] - p[3])
    if horiz == 0: return np.nan
    return (np.linalg.norm(p[1] - p[5]) + np.linalg.norm(p[2] - p[4])) / (2.0 * horiz)

def mouth_aspect_ratio(lm, w, h):
    l, r = MOUTH_CORNERS; t, b = MOUTH_VERTICAL
    pl = np.array([lm[l].x * w, lm[l].y * h]); pr = np.array([lm[r].x * w, lm[r].y * h])
    pt = np.array([lm[t].x * w, lm[t].y * h]); pb = np.array([lm[b].x * w, lm[b].y * h])
    horiz = np.linalg.norm(pl - pr)
    if horiz == 0: return np.nan
    return np.linalg.norm(pt - pb) / horiz

def euler_from_matrix(matrix):
    rot = np.array(matrix)[:3, :3]
    sy = math.sqrt(rot[0, 0] ** 2 + rot[1, 0] ** 2)
    if sy >= 1e-6:
        pitch = math.degrees(math.atan2(rot[2, 1], rot[2, 2]))
    else:
        pitch = math.degrees(math.atan2(-rot[1, 2], rot[1, 1]))
    yaw = math.degrees(math.atan2(-rot[2, 0], sy))
    return yaw, pitch

def features_desde_resultado(result, w, h):
    '''22 features geometricas a partir de un resultado de FaceLandmarker.'''
    if not result.face_landmarks:
        return np.full(len(FEATURE_NAMES), np.nan)
    lm = result.face_landmarks[0]
    ear_r = eye_aspect_ratio(lm, RIGHT_EYE, w, h)
    ear_l = eye_aspect_ratio(lm, LEFT_EYE, w, h)
    mar = mouth_aspect_ratio(lm, w, h)
    yaw, pitch = (euler_from_matrix(result.facial_transformation_matrixes[0])
                  if result.facial_transformation_matrixes else (np.nan, np.nan))
    bs = {b.category_name: b.score for b in result.face_blendshapes[0]} \
         if result.face_blendshapes else {}
    return np.array([(ear_r + ear_l) / 2.0, ear_l, ear_r, yaw, pitch, mar]
                    + [bs.get(n, np.nan) for n in BLENDSHAPES_OF_INTEREST], dtype=np.float64)

def crop_desde_resultado(frame_bgr, result, crop_size=CROP_SIZE, margin=0.25):
    '''Recorte cuadrado del rostro (identico a Fase 3.2a). Devuelve RGB uint8.'''
    if not result.face_landmarks:
        return None
    lm = result.face_landmarks[0]
    h, w = frame_bgr.shape[:2]
    xs = np.array([p.x for p in lm]) * w; ys = np.array([p.y for p in lm]) * h
    side = max(xs.max() - xs.min(), ys.max() - ys.min()) * (1 + margin)
    cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
    x0, y0 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x1, y1 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        return None
    c = cv2.resize(frame_bgr[y0:y1, x0:x1], (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(c, cv2.COLOR_BGR2RGB)

## 2. Intérpretes TFLite

Se cargan tres modelos: el extractor de apariencia y los dos modelos de decisión. Las entradas
y salidas se localizan **por su forma**, nunca por índice: el conversor de TFLite no preserva el
orden en que se declaran los `keras.Input` ni las salidas del modelo, y asumirlo produce errores
silenciosos (una salida ordinal leída como si fuera la binaria, por ejemplo).

In [ ]:
class ModeloTFLite:
    '''Envoltorio que resuelve entradas y salidas por forma, no por indice.'''
    def __init__(self, path):
        self.it = tf.lite.Interpreter(model_path=path)
        self.it.allocate_tensors()
        self.inp = self.it.get_input_details()
        self.out = self.it.get_output_details()
        self.path = path
    def _cuantizar(self, det, arr):
        if det['dtype'] == np.int8:
            sc, zp = det['quantization']
            return np.clip(np.round(arr / sc + zp), -128, 127).astype(np.int8)
        return arr.astype(det['dtype'])
    def _descuantizar(self, det, arr):
        if det['dtype'] == np.int8:
            sc, zp = det['quantization']
            return (arr.astype(np.float32) - zp) * sc
        return arr.astype(np.float32)
    def predecir(self, por_forma):
        '''por_forma: dict {(dim1, dim2, ...): array con batch 1}'''
        for d in self.inp:
            forma = tuple(int(v) for v in d['shape'][1:])
            if forma not in por_forma:
                raise KeyError(f'Falta entrada de forma {forma} en {self.path}')
            self.it.set_tensor(d['index'], self._cuantizar(d, por_forma[forma]))
        self.it.invoke()
        salidas = {}
        for d in self.out:
            forma = tuple(int(v) for v in d['shape'][1:])
            salidas[forma] = self._descuantizar(d, self.it.get_tensor(d['index']))
        return salidas
    def resumen(self):
        return {'entradas': [(tuple(int(v) for v in d['shape'][1:]), d['dtype'].__name__)
                             for d in self.inp],
                'salidas': [(tuple(int(v) for v in d['shape'][1:]), d['dtype'].__name__)
                            for d in self.out]}

extractor = ModeloTFLite(f'{MODEL_DIR}/cnn_extractor.tflite')
modelo_A   = ModeloTFLite(f'{MODEL_DIR}/modelo_A_resumen_lineal.tflite')
modelo_B   = ModeloTFLite(f'{MODEL_DIR}/modelo_B_tcn_fusion.tflite')

for nombre, m in [('extractor CNN', extractor), ('modelo A', modelo_A), ('modelo B', modelo_B)]:
    r = m.resumen()
    print(f'{nombre:15s} entradas={r["entradas"]}  salidas={r["salidas"]}')

## 3. Pipeline completo — un video, una predicción

Una sola pasada de decodificación y una sola detección de MediaPipe por frame. Los frames que
sirven a las dos ramas (las 60 posiciones de geometría y las 16 de apariencia se solapan
parcialmente) se procesan una única vez.

In [ ]:
def preparar_geometricas(seq22):
    '''NaN -> interpolacion -> medianas de entrenamiento -> deltas -> escalado.'''
    df = pd.DataFrame(seq22).interpolate(method='linear', axis=0, limit_direction='both')
    s = df.ffill().bfill().values.astype(np.float32)
    if np.isnan(s).any():
        idx = np.where(np.isnan(s)); s[idx] = np.take(GEOM_MEDIANS, idx[1])
    di = [FEATURE_NAMES.index(f) for f in DELTA_FEATURES]
    d = np.zeros((s.shape[0], len(di)), dtype=np.float32)
    d[1:] = np.diff(s[:, di], axis=0)
    s32 = np.concatenate([s, d], axis=-1)
    return ((s32 - GEOM_MEAN) / GEOM_SCALE).astype(np.float32)

def preparar_embeddings(emb):
    df = pd.DataFrame(emb).interpolate(method='linear', axis=0, limit_direction='both')
    e = df.ffill().bfill().values.astype(np.float32)
    if np.isnan(e).any():
        idx = np.where(np.isnan(e)); e[idx] = np.take(EMB_MEDIANS, idx[1])
    return ((e - EMB_MEAN) / EMB_SCALE).astype(np.float32)

def procesar_video(video_path, detector, verbose=True):
    '''Devuelve (geom_escaladas (60,32), emb_escalados (16,128), diagnostico).'''
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 0
    if total <= 0:
        cap.release(); raise ValueError(f'Video ilegible o vacio: {video_path}')

    idx_geom = np.linspace(0, total - 1, N_FRAMES).astype(int)
    idx_emb  = np.linspace(0, total - 1, N_EMB_FRAMES).astype(int)
    pos_geom, pos_emb = {}, {}
    for p, i in enumerate(idx_geom): pos_geom.setdefault(int(i), []).append(p)
    for p, i in enumerate(idx_emb):  pos_emb.setdefault(int(i), []).append(p)
    necesarios = set(pos_geom) | set(pos_emb)

    seq22 = np.full((N_FRAMES, len(FEATURE_NAMES)), np.nan)
    emb   = np.full((N_EMB_FRAMES, EMB_DIM), np.nan, dtype=np.float32)
    sin_rostro_geom = sin_rostro_emb = 0
    t0 = time.perf_counter()

    i, ultimo = 0, max(necesarios)
    while i <= ultimo:
        ret, frame = cap.read()
        if not ret: break
        if i in necesarios:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
            if i in pos_geom:                                  # rama geometrica
                f = features_desde_resultado(res, frame.shape[1], frame.shape[0])
                for p in pos_geom[i]: seq22[p] = f
                if not res.face_landmarks: sin_rostro_geom += len(pos_geom[i])
            if i in pos_emb:                                   # rama de apariencia
                crop = crop_desde_resultado(frame, res)
                if crop is None:
                    sin_rostro_emb += len(pos_emb[i])
                else:
                    x = tf.keras.applications.mobilenet_v2.preprocess_input(
                        crop.astype(np.float32))[None, ...]
                    v = extractor.predecir({(CROP_SIZE, CROP_SIZE, 3): x})
                    for p in pos_emb[i]: emb[p] = list(v.values())[0].ravel()
        i += 1
        # el frame y el crop salen de scope aca: nunca se persisten
    cap.release()

    diag = {'frames_totales': total, 'fps': round(fps, 1),
            'frames_decodificados': i, 'detecciones': len(necesarios),
            'pct_sin_rostro_geom': 100 * sin_rostro_geom / N_FRAMES,
            'pct_sin_rostro_emb': 100 * sin_rostro_emb / N_EMB_FRAMES,
            'segundos_extraccion': round(time.perf_counter() - t0, 2)}
    if verbose:
        print(f'  {diag["frames_decodificados"]} frames decodificados, '
              f'{diag["detecciones"]} detecciones | sin rostro: '
              f'{diag["pct_sin_rostro_geom"]:.0f}% geom, {diag["pct_sin_rostro_emb"]:.0f}% emb '
              f'| {diag["segundos_extraccion"]}s')
    return preparar_geometricas(seq22), preparar_embeddings(emb), diag

UMBRAL = 0.50    # decision de producto; ver la curva de umbral en metricas_modelos.json

def predecir(geom32, emb128, umbral=UMBRAL):
    entradas = {(N_FRAMES, 32): geom32[None, ...], (N_EMB_FRAMES, EMB_DIM): emb128[None, ...]}
    sa = modelo_A.predecir(entradas)
    p_A = float(sa[(1,)].ravel()[0])

    sb = modelo_B.predecir(entradas)
    p_B  = float(sb[(1,)].ravel()[0])
    ordinal = sb[(2,)].ravel()                       # P(y>0), P(y>1)
    ordinal = np.minimum.accumulate(ordinal)         # cummin: garantiza monotonicidad
    nivel   = int((ordinal > 0.5).sum())             # 0 atento, 1 leve, 2 marcada
    atencion = sb[(N_FRAMES,)].ravel()               # pesos de attention pooling

    return {'p_desatencion_A': p_A, 'clase_A': int(p_A > umbral),
            'p_desatencion_B': p_B, 'clase_B': int(p_B > umbral),
            'nivel_ordinal': nivel, 'ordinal_probs': ordinal.tolist(),
            'atencion': atencion, 'umbral': umbral}

## 4. Probar con un video

Podés subir un video propio o usar un clip de DAiSEE. Lo esperable es un video de ~10 segundos
con una cara visible, encuadre tipo webcam.

> **Nota sobre la monotonicidad ordinal.** La cabeza CORAL usa pesos independientes por umbral
> (ver la nota de diseño de Fase 3.1: la formulación con peso compartido rompe la conversión a
> TFLite). El loss ordinal empuja hacia `P(y>0) ≥ P(y>1)`, pero no lo garantiza por construcción.
> Por eso la función de predicción aplica `cummin` sobre las dos probabilidades — dos operaciones,
> costo despreciable, y la monotonicidad queda asegurada en producción.

In [ ]:
VIDEO = None    # ruta a un .avi/.mp4; None = subir uno o buscar un clip de DAiSEE

if VIDEO is None and IN_COLAB:
    candidatos = glob.glob('/root/.cache/kagglehub/**/DataSet/**/*.avi', recursive=True)
    if candidatos:
        VIDEO = candidatos[0]
        print('Usando un clip de DAiSEE ya presente en el runtime:', os.path.basename(VIDEO))
    else:
        from google.colab import files
        print('Subi un video de ~10 s con una cara visible:')
        subido = files.upload()
        VIDEO = f'/content/{list(subido)[0]}'

assert VIDEO and os.path.exists(VIDEO), 'No hay video para procesar'
print(f'\nProcesando {os.path.basename(VIDEO)} ...')

detector = create_face_landmarker()
try:
    t0 = time.perf_counter()
    geom32, emb128, diag = procesar_video(VIDEO, detector)
    t_ext = time.perf_counter() - t0
    t1 = time.perf_counter()
    r = predecir(geom32, emb128)
    t_pred = time.perf_counter() - t1
finally:
    detector.close()

etiqueta = ['Atento', 'Desatencion']
niveles  = ['Atento', 'Desatencion leve', 'Desatencion marcada']
print(f'\n{"="*58}\nRESULTADO\n{"="*58}')
print(f'Modelo A (resumen lineal)  P(desatencion) = {r["p_desatencion_A"]:.3f}'
      f'  ->  {etiqueta[r["clase_A"]]}')
print(f'Modelo B (TCN de fusion)   P(desatencion) = {r["p_desatencion_B"]:.3f}'
      f'  ->  {etiqueta[r["clase_B"]]}')
print(f'Nivel ordinal (modelo B)   {niveles[r["nivel_ordinal"]]}  '
      f'(P(y>0)={r["ordinal_probs"][0]:.3f}, P(y>1)={r["ordinal_probs"][1]:.3f})')
print(f'Umbral de decision         {r["umbral"]}')
print(f'\nTiempos: extraccion {t_ext:.2f} s | decision {t_pred*1000:.1f} ms')
print(f'Diagnostico: {diag}')

## 5. Mapa de calor temporal

Los pesos de `attention_pooling` indican en qué momentos del clip se apoyó el modelo para
decidir. Es la salida que alimenta la visualización de Sparkle: permite mostrarle al docente
*cuándo* dentro del intervalo se detectó la desatención, no solo que ocurrió.

Interpretarlos con cautela: los pesos de atención muestran dónde miró el modelo, **no** una
explicación causal de la predicción.

In [ ]:
at = r['atencion']
seg = np.arange(N_FRAMES) / N_FRAMES * 10          # clip de 10 s

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2]})
axes[0].imshow(at[None, :], aspect='auto', cmap='YlOrRd',
               extent=[0, 10, 0, 1], vmin=at.min(), vmax=at.max())
axes[0].set_yticks([]); axes[0].set_title(
    f'Mapa de calor de atencion — P(desatencion) = {r["p_desatencion_B"]:.3f} '
    f'({niveles[r["nivel_ordinal"]]})')

idx_ear = FEATURE_NAMES.index('ear_avg')
idx_jaw = FEATURE_NAMES.index('jawOpen')
axes[1].plot(seg, geom32[:, idx_ear], label='ear_avg (apertura ocular)', lw=1.4)
axes[1].plot(seg, geom32[:, idx_jaw], label='jawOpen (bostezo)', lw=1.4, alpha=0.8)
axes[1].plot(seg, at / at.max() * np.abs(geom32[:, idx_ear]).max(),
             'r--', lw=1.2, alpha=0.7, label='atencion (reescalada)')
axes[1].set_xlabel('segundos'); axes[1].set_ylabel('valor escalado')
axes[1].legend(loc='upper right', fontsize=9); axes[1].grid(alpha=0.3)
fig.tight_layout(); plt.show()

top = np.argsort(at)[-5:][::-1]
print('Momentos de mayor atencion del modelo:')
for p in top:
    print(f'  t = {p/N_FRAMES*10:4.1f} s  (peso {at[p]:.4f})')

## 6. Benchmark del pipeline completo

Dónde se va el tiempo. En este pipeline la decisión es despreciable frente a la extracción: el
costo dominante son las 60 detecciones de MediaPipe y las 16 inferencias de la CNN de apariencia.

Los números de abajo corresponden a la CPU del runtime de Colab. **No son representativos de un
dispositivo móvil** — para la tesis hace falta medirlos en el hardware objetivo real. Sirven para
comparar las etapas entre sí y ver dónde optimizar.

In [ ]:
N_REPS = 3
detector = create_face_landmarker()
try:
    tiempos = []
    for _ in range(N_REPS):
        t0 = time.perf_counter(); g, e, _ = procesar_video(VIDEO, detector, verbose=False)
        t1 = time.perf_counter(); predecir(g, e)
        t2 = time.perf_counter()
        tiempos.append((t1 - t0, t2 - t1))
finally:
    detector.close()

ext = np.mean([t[0] for t in tiempos]); dec = np.mean([t[1] for t in tiempos])

# desglose: costo de la CNN de apariencia sola
x = np.zeros((1, CROP_SIZE, CROP_SIZE, 3), dtype=np.float32)
for _ in range(5): extractor.predecir({(CROP_SIZE, CROP_SIZE, 3): x})
t0 = time.perf_counter()
for _ in range(50): extractor.predecir({(CROP_SIZE, CROP_SIZE, 3): x})
t_cnn = (time.perf_counter() - t0) / 50

df = pd.DataFrame([
    {'etapa': 'Extraccion (landmarks + CNN)', 'ms_por_clip': ext * 1000,
     'pct': ext / (ext + dec) * 100},
    {'etapa': '  de eso, CNN de apariencia', 'ms_por_clip': t_cnn * N_EMB_FRAMES * 1000,
     'pct': t_cnn * N_EMB_FRAMES / (ext + dec) * 100},
    {'etapa': 'Modelos de decision (A + B)', 'ms_por_clip': dec * 1000,
     'pct': dec / (ext + dec) * 100},
    {'etapa': 'TOTAL por clip de 10 s', 'ms_por_clip': (ext + dec) * 1000, 'pct': 100.0},
])
print(df.round(1).to_string(index=False))
print(f'\nFactor de tiempo real: {(ext+dec)/10:.3f}x '
      f'({"procesa mas rapido de lo que dura el clip" if (ext+dec) < 10 else "mas lento que tiempo real"})')

tam_total = sum(os.path.getsize(f'{MODEL_DIR}/{f}') for f in os.listdir(MODEL_DIR)
                if f.endswith('.tflite'))
print(f'\nTamano total de los .tflite en dispositivo: {tam_total/1024:.1f} kB')

## 7. Integración en Sparkle — resumen

Para embeber esto en la app hacen falta cuatro archivos de `modelo_final/`:

| Archivo | Rol |
| :--- | :--- |
| `cnn_extractor.tflite` | Rama de apariencia (se ejecuta 16 veces por clip) |
| `modelo_A_resumen_lineal.tflite` **o** `modelo_B_tcn_fusion.tflite` | Decisión |
| `preprocesamiento.npz` | Medianas y escaladores — hay que portar estos vectores a la app |
| `face_landmarker.task` | Modelo de MediaPipe (se descarga aparte) |

**Cuál de los dos modelos de decisión usar.** El A tiene mejor exactitud; el B da el nivel ordinal
y el mapa de calor. Se pueden correr los dos: sumados pesan poco y su costo es marginal frente a
la extracción de features.

**Tres cosas a resolver antes de producción:**

1. **Elegir el umbral.** Con 0,5 la tasa de falsos positivos sobre alumnos atentos es alta. La
   curva de compromiso está en `metricas_modelos.json`; la elección depende de cuánto molesta una
   alerta equivocada frente a no detectar desatención.
2. **Medir latencia en el hardware real.** Los números de acá son de una CPU de servidor.
3. **Migrar de `tf.lite.Interpreter` a `ai_edge_litert`**, que lo reemplaza a partir de TF 2.20.

**Privacidad.** El pipeline nunca escribe a disco el frame ni el recorte facial; ambos existen
solo en memoria durante la inferencia de cada frame y se descartan al salir del scope. Lo único
que persiste es el vector de decisión.